# 08 - Six-week campaign test design (DS, Day 14)

The DA plan's Day 14 task for DS: **write the 6-week test design.** The DA team builds the weekly monitoring query for its primary KPI and the audience lists.

A good test design answers, **before** the campaign runs:
1. **Which segment** is tested first, and why?
2. **Which KPI** shows it worked, and what change counts as a win?
3. **How many customers are needed** to see that change reliably, and do we have them?
4. **Who is in the campaign and who is held back** (randomised, so the comparison is fair)?
5. **What will we do with the result**, decided in advance so the answer cannot be bent afterwards?

This notebook uses the real data to answer each, and writes the design to `outputs/campaign_test_design.md`.

**Two facts from notebook 07 shape the design:**
- Buyers take **43 to 80 days** to decide, but the test runs **42 days**. A six-week test cannot measure completed purchases for buyers, so the primary KPI has to come earlier in the funnel: **making an enquiry (creating a lead)**.
- Renters decide in about **2 weeks**, so for them a completed let *can* be measured in six weeks.

## 0. Setup

In [ ]:
%pip install -q scipy

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

ROOT = Path.cwd().parent
CLEAN = ROOT / "data" / "clean"
FEAT = ROOT / "data" / "features"
OUT = ROOT / "notebooks" / "outputs"
OUT.mkdir(exist_ok=True)
for f in ["customer_segments.parquet", "customer_features.parquet", "customer_outcomes.parquet"]:
    assert (FEAT / f).exists(), f"{f} missing: run notebooks 04 to 07 first"

seg = pd.read_parquet(FEAT / "customer_segments.parquet").set_index("customer_id")
F = pd.read_parquet(FEAT / "customer_features.parquet").set_index("customer_id").loc[seg.index]
O = pd.read_parquet(FEAT / "customer_outcomes.parquet").set_index("customer_id").loc[seg.index]
cust = pd.read_parquet(CLEAN / "customers.parquet").set_index("customer_id").loc[seg.index]
leads = pd.read_parquet(CLEAN / "leads.parquet")

leads["created_date"] = pd.to_datetime(leads["created_date"])
leads["closed_date"] = pd.to_datetime(leads["closed_date"])
first_contact = pd.to_datetime(cust["first_contact_date"])
name = seg["segment_name"]
MODEL_VERSION = seg["model_version"].iloc[0]

WINDOW_DAYS = 42            # the test length: six weeks
N_WINDOWS = 8               # how many past six-week windows to measure the baseline on
DATA_END = leads["created_date"].max().normalize()
print(f"{len(seg)} customers | model {MODEL_VERSION} | leads run to {DATA_END.date()} | test length {WINDOW_DAYS} days")

## 1. Who can we reach?
Only customers who gave marketing consent can be contacted, so the test population is the consenting customers in the chosen segment.

In [ ]:
reach = pd.DataFrame({
    "customers": name.value_counts(),
    "consenting": F["consent_marketing"].astype(bool).groupby(name).sum(),
})
reach["consent_%"] = (reach["consenting"] / reach["customers"] * 100).round(1)
reach.sort_values("customers", ascending=False)

## 2. Baselines: what happens in a normal six weeks?
We measure two things over the last 8 six-week windows in the data, for each segment. In every window, only customers who already existed at the start (first contact before the window) count.
- **Enquiry rate (primary KPI for buyers):** share of customers who created at least one new lead in the window.
- **Won-in-window rate:** share who created a lead in the window **and** had it won before the window ended. Only meaningful where decisions are quick (renters).

A baseline measured over several windows shows how much it moves by chance and with the season. That spread decides how large a lift would be believable.

In [ ]:
rows = []
for w in range(N_WINDOWS):
    end = DATA_END - pd.Timedelta(days=WINDOW_DAYS * w)
    start = end - pd.Timedelta(days=WINDOW_DAYS)
    existing = first_contact < start
    in_win = leads[(leads["created_date"] >= start) & (leads["created_date"] < end)]
    enq = set(in_win["customer_id"])
    won = set(in_win.loc[(in_win["lead_stage"] == "Won") & (in_win["closed_date"] <= end), "customer_id"])
    for s, idx in name.groupby(name).groups.items():
        elig = existing.loc[idx][existing.loc[idx]].index
        rows.append({"window_end": end, "segment": s, "eligible": len(elig),
                     "enquiry_rate": np.mean([c in enq for c in elig]) if len(elig) else np.nan,
                     "won_in_window_rate": np.mean([c in won for c in elig]) if len(elig) else np.nan})
win = pd.DataFrame(rows)
base = win.groupby("segment").agg(eligible=("eligible", "mean"),
                                  enquiry_rate_mean=("enquiry_rate", "mean"), enquiry_rate_sd=("enquiry_rate", "std"),
                                  won_rate_mean=("won_in_window_rate", "mean"), won_rate_sd=("won_in_window_rate", "std"))
base[["enquiry_rate_mean", "enquiry_rate_sd", "won_rate_mean", "won_rate_sd"]] *= 100
base.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
for s, g in win.groupby("segment"):
    ax.plot(g["window_end"], g["enquiry_rate"] * 100, marker="o", label=s)
ax.set(title="Enquiry rate per six-week window, by segment", xlabel="window end", ylabel="% of customers creating a lead")
ax.legend(fontsize=7, ncol=2); plt.tight_layout(); plt.savefig(OUT / "test_baseline_enquiry_rate.png", dpi=150); plt.show()

## 3. Power: how many customers do we need?
**Power** is the chance the test detects a real effect. We use the standard choice: **80% power, 5% false-alarm rate (two-sided)**.
- `n_needed` = customers per arm to detect a given **relative lift** (for example +20% means an enquiry rate of 1.0% rising to 1.2%).
- `MDE` (minimum detectable effect) = the smallest relative lift the test could reliably detect *with the customers we have* (half the consenting customers in each arm).
A small baseline rate means a large sample is needed: this is the main constraint on what can be tested.

In [ ]:
ALPHA, POWER = 0.05, 0.80
z_a, z_b = norm.ppf(1 - ALPHA / 2), norm.ppf(POWER)

def n_per_arm(p0, lift):
    p1 = p0 * (1 + lift)
    pbar = (p0 + p1) / 2
    return ((z_a * np.sqrt(2 * pbar * (1 - pbar)) + z_b * np.sqrt(p0 * (1 - p0) + p1 * (1 - p1))) ** 2) / (p1 - p0) ** 2

def mde(p0, n):
    lo, hi = 0.001, 10.0
    if n_per_arm(p0, hi) > n:
        return np.inf
    for _ in range(60):
        mid = (lo + hi) / 2
        lo, hi = (mid, hi) if n_per_arm(p0, mid) > n else (lo, mid)
    return hi

tbl = reach.join(base[["enquiry_rate_mean"]])
tbl["baseline_enquiry_%"] = tbl["enquiry_rate_mean"].round(2)
p0 = tbl["enquiry_rate_mean"] / 100
tbl["per_arm"] = (tbl["consenting"] / 2).astype(int)
tbl["MDE_lift_%"] = [int(np.ceil(mde(p, n) * 100)) if p > 0 else np.inf for p, n in zip(p0, tbl["per_arm"])]
tbl["n_needed_for_+20%"] = [int(np.ceil(n_per_arm(p, 0.20))) if p > 0 else np.nan for p in p0]
tbl["can_detect_+20%"] = tbl["per_arm"] >= tbl["n_needed_for_+20%"]
win_rate = (O.groupby(name)["n_won"].sum() / O.groupby(name)["n_leads_total"].sum())
tbl["win_rate_%"] = (win_rate * 100).round(1)
tbl["extra_enquiries_at_+20%"] = (tbl["per_arm"] * p0 * 0.20).round(0)
tbl["extra_wins_at_+20%"] = (tbl["extra_enquiries_at_+20%"] * win_rate).round(1)
tbl[["customers", "consenting", "baseline_enquiry_%", "per_arm", "MDE_lift_%", "n_needed_for_+20%", "can_detect_+20%", "win_rate_%", "extra_enquiries_at_+20%", "extra_wins_at_+20%"]].sort_values("MDE_lift_%")

## 4. Choose the test segment
The rule, written down before looking at who "wins": among segments where the test is **powered to detect a lift of at most `MAX_MDE`**, pick the one with the most **expected extra wins** at a +20% lift. That balances "can we measure it" with "does it matter to the business".
You can override with `TEST_SEGMENT`, but write down why.

In [ ]:
MAX_MDE = 40                 # CONFIG: largest relative lift (%) we accept as the smallest effect the test can detect.
                             # On the real data no segment gets below 37% at six weeks, so 35 stops the notebook; 40 is a deliberate choice, see the design file.
TEST_SEGMENT = None          # CONFIG: set a segment name to override the rule

eligible = tbl[tbl["MDE_lift_%"] <= MAX_MDE]
if TEST_SEGMENT is None:
    assert len(eligible), (f"No segment can detect a lift of {MAX_MDE}% or less (the best is {tbl['MDE_lift_%'].min():.0f}% for {tbl['MDE_lift_%'].idxmin()}). "
                           "Options: raise MAX_MDE and accept that only a bigger lift can be detected, set TEST_SEGMENT by hand, or run a longer test.")
    TEST_SEGMENT = eligible["extra_wins_at_+20%"].idxmax()
    print(f"Rule picked: {TEST_SEGMENT}")
else:
    print(f"Manual choice: {TEST_SEGMENT}")
display(eligible[["customers", "consenting", "baseline_enquiry_%", "MDE_lift_%", "extra_wins_at_+20%"]].sort_values("extra_wins_at_+20%", ascending=False))

## 5. Randomise: who gets the campaign and who is held back
Consenting customers in the test segment are split **50/50 at random** (fixed seed so it can be reproduced). The control group gets business as usual, so any difference afterwards is due to the campaign and not to who we picked.
The balance check compares the two groups on things that could affect the result: a fair split should show differences close to zero.

In [ ]:
SEED = 2026
pool = seg[(name == TEST_SEGMENT) & F["consent_marketing"].astype(bool)].index
rng = np.random.default_rng(SEED)
order = rng.permutation(len(pool))
arm = pd.Series("control", index=pool)
arm.iloc[order[: len(pool) // 2]] = "treatment"
arm.name = "arm"

check = pd.DataFrame({
    "budget_gbp": F.loc[pool, "budget_gbp"],
    "n_sessions": F.loc[pool, "n_sessions"],
    "n_leads": F.loc[pool, "n_leads"],
    "has_lead": (F.loc[pool, "n_leads"] > 0).astype(float),
    "mobile_share": F.loc[pool, "share_mobile"],
    "overseas": F.loc[pool, "is_overseas"].astype(float),
})
g = check.groupby(arm)
smd = ((g.mean().loc["treatment"] - g.mean().loc["control"]) / np.sqrt((g.var().loc["treatment"] + g.var().loc["control"]) / 2)).round(3)
bal = g.mean().T.round(3)
bal["std_diff"] = smd
print(f"{len(pool)} consenting customers in '{TEST_SEGMENT}': {int((arm=='treatment').sum())} treatment, {int((arm=='control').sum())} control")
print("Standardised difference (should be near 0; above about 0.1 would be a concern):")
bal

In [ ]:
assignment = pd.DataFrame({"customer_id": pool, "segment_name": TEST_SEGMENT, "arm": arm.values,
                           "model_version": MODEL_VERSION, "seed": SEED})
assignment.to_parquet(FEAT / "test_assignment.parquet", index=False)
assignment.to_csv(OUT / "test_assignment.csv", index=False)
print("Saved", len(assignment), "rows ->", OUT / "test_assignment.csv")

## 6. Write the design
The design is written with the real numbers. The **decision rule is fixed now**, before any result exists.

In [ ]:
row = tbl.loc[TEST_SEGMENT]
n_arm = int((arm == "treatment").sum())
p0_ = row["baseline_enquiry_%"] / 100
sd_ = base.loc[TEST_SEGMENT, "enquiry_rate_sd"] / 100
mde_exact = mde(p0_, n_arm)
mde_pct = int(np.ceil(mde_exact * 100))
win_lift = max(0.20, mde_pct / 100)
needed_win = int(np.ceil(n_per_arm(p0_, win_lift)))
days = O.loc[seg.index[name == TEST_SEGMENT], "days_to_decision"].median()
balance_txt = "are balanced" if smd.abs().max() < 0.1 else "are NOT well balanced (re-run with a different SEED before using this design)"

design = f'''# Six-week campaign test design ({MODEL_VERSION})

Written by the DS team before the test runs. Numbers come from the cleaned data up to {DATA_END.date()}.

## 1. What we are testing
**Segment:** {TEST_SEGMENT} ({int(row['customers']):,} customers; {int(row['consenting']):,} gave marketing consent).
**Why this segment:** it was chosen by a rule written in advance. Among segments where the test can detect a lift of {MAX_MDE}% or less, it has the most expected extra wins: about {row['extra_wins_at_+20%']:.0f} extra completed deal(s) if enquiries rise by 20% ({row['extra_enquiries_at_+20%']:.0f} extra enquiries at a {row['win_rate_%']:.0f}% win rate).
**Treatment:** the campaign planned for next month, sent to the treatment arm only. **Control:** business as usual, no campaign.

## 2. How customers were assigned
Consenting customers in the segment were split 50/50 at random (seed {SEED}): **{n_arm:,} treatment and {int((arm == 'control').sum()):,} control**. The groups {balance_txt} on budget, browsing, past leads, mobile use and overseas share (largest standardised difference {smd.abs().max():.2f}; below 0.1 is balanced). The assignment is in `test_assignment.csv`.

## 3. The primary KPI and what counts as a win
**Primary KPI: enquiry rate** = the share of customers in each arm who create at least one new lead during the six weeks.
- Why not completed sales? The median decision for this segment takes {days:.0f} days, and the test lasts {WINDOW_DAYS}. A sale cannot be counted in time unless decisions are quick, so the enquiry is the measurable first step.
- **Baseline:** {p0_ * 100:.2f}% per six weeks (typical movement between six-week windows: +/- {sd_ * 100:.2f} points).
- **A win:** the treatment enquiry rate is at least **{win_lift * 100:.0f}% higher** than control (that is, at least {p0_ * (1 + win_lift) * 100:.2f}% against {p0_ * 100:.2f}%) **and** a two-sided test gives p < {ALPHA}.
- **Sample size:** detecting a {win_lift * 100:.0f}% lift needs about {needed_win:,} customers per arm. We have {n_arm:,}, so the smallest lift the test can reliably detect is about {mde_pct}%.

## 4. Secondary measures (reported, not used to declare a win)
- Leads that reach a viewing, and completed deals in the six weeks (for quick-deciding customers).
- Segment mix of the enquiries that came in.
- Guardrails for the DA team to monitor weekly: unsubscribe and complaint rates in the treatment arm. If either rises sharply, pause the campaign and report.

## 5. Rules so the result can be trusted
- Fixed length of six weeks. **No stopping early because the numbers look good.** The weekly monitoring is for guardrails and delivery problems only.
- One primary test, two-sided, at the {ALPHA * 100:.0f}% level. Other cuts of the data are reported as exploratory.
- Treat the control group exactly as normal. No other campaign may target either arm differently.
- If the lift is below the win threshold, report it as **inconclusive**, not as "the campaign does not work": a test of this size can only detect a lift of about {mde_pct}% or more, so a smaller real effect could be missed. Report the size of the effect and its range.

## 6. After the test
- **Win:** roll the campaign out to the rest of the segment, then test the next segment in the ranking.
- **No win:** keep the learning, review the message or channel, and do not repeat the same test unchanged.
- Re-score the segments monthly (see the refresh process) so the audience stays current.
'''
(OUT / "campaign_test_design.md").write_text(design, encoding="utf-8")
print(design)

## 7. Validate

In [ ]:
results = {
    "test segment has a name that exists in customer_segments": TEST_SEGMENT in set(name),
    "every assigned customer gave marketing consent": bool(F.loc[assignment['customer_id'], 'consent_marketing'].astype(bool).all()),
    "every assigned customer is in the test segment": bool((name.loc[assignment['customer_id']] == TEST_SEGMENT).all()),
    "arms are the same size (within 1 customer)": abs(int((arm == 'treatment').sum()) - int((arm == 'control').sum())) <= 1,
    "arms are balanced (largest standardised difference < 0.1)": bool(smd.abs().max() < 0.1),
    "test can detect the win threshold with the customers available": bool(n_arm >= needed_win),
    "design file written": (OUT / "campaign_test_design.md").exists(),
}
for k_, v in results.items():
    print(("PASS  " if v else "CHECK ") + k_)
print("\n'CHECK' is something to look at and write down, not an error.")

## Hand-off notes
- **To the DA team (Day 14):** `outputs/test_assignment.csv` (who is treatment and who is control) and `outputs/campaign_test_design.md`. They build the weekly monitoring query for the **primary KPI (enquiry rate)** and the guardrails, and add `target_segment` to next month's campaign table.
- **Important for DA:** the control group must **not** be sent the campaign. Exclude control customers from the audience lists.
- **Write down:** why this segment was chosen (the rule), the smallest lift the test can detect, and that completed sales cannot be measured in six weeks for buyers.
- **Next (Day 15):** scoring new customers and the monthly refresh, plus the segment-size drift alert.